In [25]:
import polars as pl
from pathlib import Path


In [26]:
path = Path("all_preds.xlsx")
if not path.exists():
    path = Path("data") / "all_preds.xlsx"


In [27]:
df = pl.read_excel("/home/vandy/Work/DASC5309/scot-forge/data/silver/all_preds.xlsx", sheet_name="Basic Xgboost")

In [28]:
df.head()

Date,Usage - 1,Usage - 2,Usage - 2_1,Actual_Total,Predicted_Total
date,f64,f64,f64,f64,f64
2024-01-01,323.95,1355.36,28.22,1707.53,2156.251
2024-01-02,383.52,2181.96,20.9,2586.38,2587.1426
2024-01-03,400.24,2065.96,26.12,2492.32,2537.8406
2024-01-04,409.64,2272.88,29.26,2711.78,2482.2034
2024-01-05,328.13,1438.96,26.12,1793.21,1826.7349


In [29]:
import polars as pl


def mae(df: pl.DataFrame, y: str, yhat: str):
    return df.select((pl.col(y) - pl.col(yhat)).abs().mean()).item()


def mse(df: pl.DataFrame, y: str, yhat: str):
    return df.select(((pl.col(y) - pl.col(yhat)) ** 2).mean()).item()


def rmse(df: pl.DataFrame, y: str, yhat: str):
    return df.select(((pl.col(y) - pl.col(yhat)) ** 2).mean().sqrt()).item()


def mape(df: pl.DataFrame, y: str, yhat: str):
    return df.select(((pl.col(y) - pl.col(yhat)).abs() / pl.col(y).abs()).mean() * 100).item()


def wmape(df: pl.DataFrame, y: str, yhat: str):
    return df.select(((pl.col(y) - pl.col(yhat)).abs().sum() / pl.col(y).abs().sum()) * 100).item()


def smape(df: pl.DataFrame, y: str, yhat: str):
    return df.select(
        ((2 * (pl.col(yhat) - pl.col(y)).abs()) / (pl.col(y).abs() + pl.col(yhat).abs())).mean() * 100
    ).item()

In [30]:
df.select(
    (pl.col("Actual_Total") - pl.col("Predicted_Total")).abs().mean().alias("MAE"),
    (pl.col("Actual_Total") - pl.col("Predicted_Total")).pow(2).mean().alias("MSE"),
    ((pl.col("Actual_Total") - pl.col("Predicted_Total")) ** 2).mean().sqrt().alias("RMSE"),
    (((pl.col("Actual_Total") - pl.col("Predicted_Total")).abs() / pl.col("Actual_Total").abs()).mean() * 100).alias(
        "MAPE"
    ),
    (
        ((pl.col("Actual_Total") - pl.col("Predicted_Total")).abs().sum() / pl.col("Actual_Total").abs().sum()) * 100
    ).alias("WMAPE"),
    (
        (
            (2 * (pl.col("Predicted_Total") - pl.col("Actual_Total")).abs())
            / (pl.col("Actual_Total").abs() + pl.col("Predicted_Total").abs())
        ).mean()
        * 100
    ).alias("SMAPE"),
)

MAE,MSE,RMSE,MAPE,WMAPE,SMAPE
f64,f64,f64,f64,f64,f64
219.774168,83997.739315,289.823635,10.521465,10.332617,10.335398


In [ ]:
USAGE_COLUMNS = ["Usage - 1", "Usage - 2", "Usage - 2_1"]


# Read every model sheet from the workbook
workbook = pl.read_excel("/home/vandy/Work/DASC5309/scot-forge/data/silver/all_preds.xlsx", sheet_id=0)


# Unpivot each sheet to: Date, Usage Type, Actual, Predicted
unpivoted_by_model: dict[str, pl.DataFrame] = {}
for model_name, model_df in workbook.items():
    actual_long = (
        model_df.select(["Date", *USAGE_COLUMNS])
        .unpivot(
            index=["Date"],
            on=USAGE_COLUMNS,
            variable_name="Usage Type",
            value_name="Actual",
        )
        .with_columns(pl.col("Actual").cast(pl.Float64))
    )

    pred_cols = [f"Pred_{usage}" for usage in USAGE_COLUMNS]
    if all(col in model_df.columns for col in pred_cols):
        predicted_long = (
            model_df.select(["Date", *pred_cols])
            .rename(dict(zip(pred_cols, USAGE_COLUMNS, strict=True)))
            .unpivot(
                index=["Date"],
                on=USAGE_COLUMNS,
                variable_name="Usage Type",
                value_name="Predicted",
            )
            .with_columns(
                pl.col("Predicted")
                .cast(pl.Utf8)
                .str.replace_all(",", "")
                .cast(pl.Float64, strict=False)
            )
        )
    else:
        predicted_long = actual_long.select(["Date", "Usage Type"]).with_columns(
            pl.lit(None, dtype=pl.Float64).alias("Predicted")
        )

    tidy_sheet = (
        actual_long.join(predicted_long, on=["Date", "Usage Type"], how="left")
        .select(["Date", "Usage Type", "Actual", "Predicted"])
        .sort(["Date", "Usage Type"])
    )

    unpivoted_by_model[model_name] = tidy_sheet


# Optional combined long data with model label for downstream evaluation
long_all = pl.concat(
    [
        df.with_columns(pl.lit(model_name).alias("Model"))
        for model_name, df in unpivoted_by_model.items()
    ],
    how="vertical_relaxed",
).select(["Model", "Date", "Usage Type", "Actual", "Predicted"])


# Evaluate each model by usage type (3 rows per model)
metrics_by_usage = (
    long_all.with_columns(
        (pl.col("Actual") - pl.col("Predicted")).abs().alias("abs_error"),
        (pl.col("Actual") - pl.col("Predicted")).pow(2).alias("sq_error"),
        (pl.col("Actual") - pl.col("Predicted")).abs().pow(3).alias("L3_error"),
        pl.col("Actual").abs().alias("abs_actual"),
        (pl.col("Actual").abs() + pl.col("Predicted").abs()).alias("smape_denom"),
    )
    .group_by(["Model", "Usage Type"])
    .agg(
        pl.col("abs_error").mean().alias("MAE"),
        pl.col("sq_error").mean().alias("MSE"),
        pl.col("L3_error").mean().alias("L3"),
        pl.col("sq_error").mean().sqrt().alias("RMSE"),
        pl.when(pl.col("abs_actual") > 0)
        .then((pl.col("abs_error") / pl.col("abs_actual")) * 100)
        .otherwise(None)
        .mean()
        .alias("MAPE"),
        ((pl.col("abs_error").sum() / pl.col("abs_actual").sum()) * 100).alias("WMAPE"),
        pl.when(pl.col("smape_denom") > 0)
        .then((2 * pl.col("abs_error") / pl.col("smape_denom")) * 100)
        .otherwise(None)
        .mean()
        .alias("SMAPE"),
    )
    .sort(["Model", "Usage Type"])
)


# Check row count per model (should be 3 usage types each)
rows_per_model = metrics_by_usage.group_by("Model").len().sort("Model")


unpivoted_by_model["LGBM + Temp"].head(), metrics_by_usage, rows_per_model

(shape: (5, 4)
 ┌────────────┬─────────────┬─────────┬─────────────┐
 │ Date       ┆ Usage Type  ┆ Actual  ┆ Predicted   │
 │ ---        ┆ ---         ┆ ---     ┆ ---         │
 │ date       ┆ str         ┆ f64     ┆ f64         │
 ╞════════════╪═════════════╪═════════╪═════════════╡
 │ 2024-01-01 ┆ Usage - 1   ┆ 323.95  ┆ 313.831235  │
 │ 2024-01-01 ┆ Usage - 2   ┆ 1355.36 ┆ 1843.756527 │
 │ 2024-01-01 ┆ Usage - 2_1 ┆ 28.22   ┆ 27.821462   │
 │ 2024-01-02 ┆ Usage - 1   ┆ 383.52  ┆ 372.472668  │
 │ 2024-01-02 ┆ Usage - 2   ┆ 2181.96 ┆ 2214.836815 │
 └────────────┴─────────────┴─────────┴─────────────┘,
 shape: (24, 9)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ Model     ┆ Usage     ┆ MAE       ┆ MSE       ┆ … ┆ RMSE      ┆ MAPE      ┆ WMAPE     ┆ SMAPE    │
 │ ---       ┆ Type      ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
 │ str       ┆ ---       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f

In [32]:
metrics_by_usage.write_csv("/home/vandy/Work/DASC5309/scot-forge/data/silver/metrics_by_usage.csv")